# Preparation WISE soil data

In [ ]:
# Libraries
import os, time, shutil
import numpy as np
import xarray as xr
import rioxarray
import geopandas as gpd
import rasterio
from rasterio import features
from  scipy import ndimage
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir_data =  '../data/'
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'

---

### Pre-processing

In [ ]:
# Function to trim memory of workers
def trim_memory() -> int:
    import ctypes
    libc = ctypes.CDLL("libc.so.6")
    return libc.malloc_trim(0)

In [ ]:
# Prepare wise soc data
def prep_wise():

    """Prepare Wise data and export as tif"""
    
    # Get wise shapefile
    shp = gpd.read_file(f'{dir_data}wise30sec/HWs30PRO.shp')

    # Get target tif
    rst = rasterio.open(f'{dir_data}worldclim/wc2.1_30s_elev/wc2.1_30s_elev.tif')

    # Create 0-30, 30-100, 100-200cm depth layers
    shp['wise_soc_0-30cm'] = (shp['SOC_P1'] * 10)
    shp['wise_soc_30-100cm'] = ((shp['SOC_P3'] - shp['SOC_P1']) * 10)
    shp['wise_soc_100-200cm'] = ((shp['SOC_P5'] - shp['SOC_P3']) * 10)

    # Change dtype of soc columns
    for i in ['wise_soc_0-30cm', 'wise_soc_30-100cm', 'wise_soc_100-200cm']:
        shp[i] = shp[i].round(0).astype('int16')

    # Reproject
    shp = shp.to_crs("EPSG:4326")

    # Get meta data from target raster
    meta = rst.meta.copy()
    meta.update(compress='lzw')

    # Rasterize and export
    # https://gis.stackexchange.com/questions/151339/rasterize-a-shapefile-with-geopandas-or-fiona-python
    for i in ['wise_soc_0-30cm', 'wise_soc_30-100cm', 'wise_soc_100-200cm']:
        out_fn = f'{dir01}{i}.tif'

        with rasterio.open(out_fn, 'w+', **meta) as out:
            out_arr = out.read(1)

            shapes = ((geom,value) for geom, value in zip(shp.geometry, shp[i]))

            burned = features.rasterize(shapes=shapes, 
                                        fill=0, 
                                        out=out_arr, 
                                        transform=out.transform)
            out.write_band(1, burned)


        # Get raster
        da = rioxarray.open_rasterio(f'{dir01}{i}.tif')
        # Replace fillvalue with nan
        da = da.where(da != -32768)
        # Add attribute long_name
        da.attrs['long_name'] = i
        # Select area and export
        da \
            .sel(y=slice(80, -60)) \
            .rio.to_raster(f'{dir01}{i}.tif')

%time prep_wise()

---

### Fill land nans

In [ ]:
def fill_nans(var, dir_out):
    """
    Fills NaN values in the specified variable's dataset using the nearest valid 
    data, applies a land mask, and exports the result to a new Zarr dataset.

    Args:
        var (str): Name of the variable to process (e.g., 'temperature', 'precipitation').
        dir_out (str): Directory where prepared data is stored and the filled dataset will be exported.

    Returns:
        None
    """

    def fill_nans_array(data, invalid):
        """
        Replace invalid (NaN) data cells by the value of the nearest valid data 
        cell.
        """
        ind = ndimage.distance_transform_edt(invalid,
                                             return_distances=False,
                                             return_indices=True)
        return data[tuple(ind)]

    # Paths for input and output
    land_mask_path = os.path.join(dir_out, 'ds_prep_copernicus_land_mask_r001.zarr')
    var_data_path = os.path.join(dir_out, f'ds_regridded_{var}.zarr')
    output_path = os.path.join(dir_out, f'ds_prep_{var}_r001.zarr')

    # Read land mask data
    da_land = xr.open_zarr(land_mask_path) \
                .chunk(dict(lat=5000, lon=5000)) \
                .copernicus_land_mask \
                .compute()

    # Read variable data
    da_var = rioxarray.open_rasterio(dir01 + var + '.tif', 
                                     chunks=dict(y=5000, x=5000))[0] \
        .rename(y='lat', x='lon') \
        .sel(lat=slice(da_land.lat.max(), da_land.lat.min()),
             lon=slice(da_land.lon.min(), da_land.lon.max()))

    # Fill nan using function fill_nans_array
    # If there are no NaNs, skip filling process
    if not da_var.isnull().any():
        da_fill = da_var.values  # No filling required
    else:
        da_fill = fill_nans_array(da_var.values, da_var.isnull().values)

    # Create a new Dataset with filled data
    ds_filled = xr.Dataset(dict(lat = da_var.lat, lon=da_var.lon))
    ds_filled[var] = (('lat', 'lon'), da_fill)

    # Apply land mask to the filled data
    ds_filled = ds_filled.where(da_land)

    # Export the filled dataset to Zarr format
    ds_filled \
        .drop_vars(['band', 'spatial_ref']) \
        .chunk(dict(lat=5000, lon=5000)) \
             .to_zarr(output_path, mode='w')


# List of files
list_wise = ['wise_soc_0-30cm', 'wise_soc_30-100cm', 'wise_soc_100-200cm']

# Fill nans
for i in list_wise:
    print(i)
    %time fill_nans(i, dir01)

---

### Post-processing

In [ ]:
xr.open_mfdataset(f'{dir01}ds_prep_wise_*_*_r001.zarr', engine='zarr') \
    .to_zarr(f'{dir01}ds_prep_wise_r001.zarr', mode='w');

In [ ]:
for i in ['wise_soc_0-30cm', 'wise_soc_30-100cm', 'wise_soc_100-200cm']:
    shutil.rmtree(f'{dir01}ds_prep_{i}_r001.zarr')